<a href="https://colab.research.google.com/github/Hassan-Himaz/GM1-Crutch/blob/main/Advanced_Analysis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Loading the Data Functions

In [ ]:
import sqlite3
import math

def get_patient_daily_data(patient_db_path, meta_db_path, patient_id, date_str):
    """
    Loads all session data for a patient on a specific day and retrieves metadata.
    """
    # 1. Get Metadata
    conn_meta = sqlite3.connect(meta_db_path)
    meta_query = "SELECT age, gender, body_weight FROM patients WHERE patient_id = ?"
    meta_df = pd.read_sql_query(meta_query, conn_meta, params=(patient_id,))
    conn_meta.close()

    if meta_df.empty:
        return None, None

    # 2. Get Session Data
    # Assuming sessions are identified by date in the patient-specific DB
    all_sessions = []
    with CrutchWalkDB(patient_db_path) as db:
        # Assuming we need to find how many sessions exist or iterate indices
        # For this example, we'll assume a loop or a specific search method exists
        # Here we simulate loading multiple sessions for that date
        idx = 0
        while True:
            try:
                df_sess = db.load_session_samples(patient_id, date_str, session_index=idx)
                if df_sess is None or df_sess.empty: break
                df_sess['session_id'] = idx
                all_sessions.append(df_sess)
                idx += 1
            except:
                break

    if not all_sessions:
        return None, meta_df.iloc[0]

    return pd.concat(all_sessions, ignore_index=True), meta_df.iloc[0]

# Step Detection (Get Updated From Riya)

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt


def detect_crutch_steps(
    csv_path,
    body_mass_kg=70,
    force_col="force",
    time_col="time",
    threshold_fraction=0.05,
    min_duration=0.15,
    max_duration=3.0,
    smoothing_window=5,
):
    df = pd.read_csv(csv_path)

    for col in [time_col, force_col]:
        if col not in df.columns:
            raise ValueError(f"Missing required column: {col}")

    df = df.sort_values(time_col).reset_index(drop=True)

    df["force_smooth"] = (
        df[force_col]
        .rolling(window=smoothing_window, center=True, min_periods=1)
        .mean()
    )

    if {"ax", "ay", "az"}.issubset(df.columns):
        df["acc_mag"] = np.sqrt(df["ax"]**2 + df["ay"]**2 + df["az"]**2)
    else:
        df["acc_mag"] = np.nan

    if {"gx", "gy", "gz"}.issubset(df.columns):
        df["gyro_mag"] = np.sqrt(df["gx"]**2 + df["gy"]**2 + df["gz"]**2)
    else:
        df["gyro_mag"] = np.nan

    threshold_n = threshold_fraction * body_mass_kg * 9.81
    above = df["force_smooth"] > threshold_n

    events = []
    in_event = False
    start_idx = None

    for i, is_above in enumerate(above):
        if is_above and not in_event:
            start_idx = i
            in_event = True

        elif not is_above and in_event:
            end_idx = i - 1
            in_event = False

            event_df = df.iloc[start_idx:end_idx + 1]
            start_time = event_df[time_col].iloc[0]
            end_time = event_df[time_col].iloc[-1]
            duration = end_time - start_time

            if duration < min_duration or duration > max_duration:
                continue

            peak_force = event_df["force_smooth"].max()
            peak_idx = event_df["force_smooth"].idxmax()
            peak_time = df.loc[peak_idx, time_col]

            impulse = np.trapezoid(
                event_df["force_smooth"],
                event_df[time_col]
            )

            events.append({
                "start_idx": start_idx,
                "end_idx": end_idx,
                "start_time": start_time,
                "end_time": end_time,
                "peak_time": peak_time,
                "duration_s": duration,
                "peak_force_n": peak_force,
                "impulse_ns": impulse,
                "peak_acc": event_df["acc_mag"].max(),
                "peak_gyro": event_df["gyro_mag"].max(),
            })

    if in_event:
        end_idx = len(df) - 1
        event_df = df.iloc[start_idx:end_idx + 1]
        start_time = event_df[time_col].iloc[0]
        end_time = event_df[time_col].iloc[-1]
        duration = end_time - start_time

        if min_duration <= duration <= max_duration:
            peak_force = event_df["force_smooth"].max()
            peak_idx = event_df["force_smooth"].idxmax()
            peak_time = df.loc[peak_idx, time_col]

            impulse = np.trapezoid(
                event_df["force_smooth"],
                event_df[time_col]
            )

            events.append({
                "start_idx": start_idx,
                "end_idx": end_idx,
                "start_time": start_time,
                "end_time": end_time,
                "peak_time": peak_time,
                "duration_s": duration,
                "peak_force_n": peak_force,
                "impulse_ns": impulse,
                "peak_acc": event_df["acc_mag"].max(),
                "peak_gyro": event_df["gyro_mag"].max(),
            })

    events_df = pd.DataFrame(events)
    return df, events_df, threshold_n


def plot_detected_steps(df, events_df, threshold_n, time_col="time"):
    plt.figure(figsize=(12, 5))
    plt.plot(df[time_col], df["force_smooth"], label="Smoothed force")
    plt.axhline(threshold_n, linestyle="--", label="5% body-weight threshold")

    for _, event in events_df.iterrows():
        plt.axvspan(event["start_time"], event["end_time"], alpha=0.2)

    plt.xlabel("Time (s)")
    plt.ylabel("Force (N)")
    plt.title("Detected crutch step events")
    plt.legend()
    plt.show()


# For Google Colab: upload your CSV first
from google.colab import files
uploaded = files.upload()

csv_filename = list(uploaded.keys())[0]

df, steps, threshold = detect_crutch_steps(
    csv_filename,
    body_mass_kg=70
)

print(f"Threshold used: {threshold:.2f} N")
print("\nDetected steps:")
print(steps)

plot_detected_steps(df, steps, threshold)

KeyboardInterrupt: 

# Wrist Strain

In [ ]:
def get_RSI_params(df, meta_row, time_col="time", force_col="force", P=47):
    R_max_table = {
        "Male": {"18-24": 0.4897, "25-34": 0.4597, "35-44": 0.4141, "45-54": 0.3871, "55-64": 0.3732},
        "Female": {"18-24": 0.3870, "25-34": 0.4042, "35-44": 0.3722, "45-54": 0.3367, "55-64": 0.2904}
    }

    # 1. Extract Metadata
    age = meta_row['age']
    gender = meta_row['gender']
    BW = meta_row['body_weight']

    if age < 25: age_cat = "18-24"
    elif age < 35: age_cat = "25-34"
    elif age < 45: age_cat = "35-44"
    elif age < 55: age_cat = "45-54"
    else: age_cat = "55-64"

    r_max_val = R_max_table[gender][age_cat]

    # 2. Calculate Session & Step Metrics
    total_h = 0
    total_stance_duration_s = 0
    total_steps = 0
    total_force = 0
    force_count = 0

    for sess_id in df['session_id'].unique():
        sess_df = df[df['session_id'] == sess_id].sort_values(time_col)

        # Session duration
        duration_s = sess_df[time_col].iloc[-1] - sess_df[time_col].iloc[0]
        total_h += (duration_s / 3600)

        if 'step_index' in sess_df.columns:
            step_indices = sorted(sess_df['step_index'].unique())
            # Drop only first and last step (2 in total)
            if len(step_indices) > 2:
                analyzed_steps = step_indices[1:-1]
                sess_df = sess_df[sess_df['step_index'].isin(analyzed_steps)]

                total_steps += len(analyzed_steps)
                stance_df = sess_df[sess_df['Step Phase'].str.lower() == 'stance']

                if not stance_df.empty:
                    step_durations = stance_df.groupby('step_index')[time_col].agg(lambda x: x.max() - x.min())
                    total_stance_duration_s += step_durations.sum()
                    total_force += stance_df[force_col].sum()
                    force_count += len(stance_df)

    # 3. Final RSI Params
    avg_F = total_force / force_count if force_count > 0 else 0
    I = avg_F / (BW * 9.81 * r_max_val)
    I = min(max(I, 0.001), 1.0)

    E = total_steps / (total_h * 60) if total_h > 0 else 0
    D = total_stance_duration_s / total_steps if total_steps > 0 else 0
    H = total_h

    return [I, E, D, H, P]

In [ ]:
# def get_RSI_params(df, meta_df = (age, gender) day, time_col="time", force_col="force", scaling = 1.0, P=47):
#   R_max = {
#     "Male": {
#         "18-24": 0.4897,
#         "25-34": 0.4597,
#         "35-44": 0.4141,
#         "45-54": 0.3871,
#         "55-64": 0.3732
#     },
#     "Female": {
#         "18-24": 0.3870,
#         "25-34": 0.4042,
#         "35-44": 0.3722,
#         "45-54": 0.3367,
#         "55-64": 0.2904
#     }
#   }
#   # extract age_category from metadata, if below 18 say 18-24, and if above 64 say
#   # extract gender

#   # for each session in day:
#   #   store the length of step index to get step count
#   #   find out start and finish time to get the duration
#   #   for each step_index:
#   #     filter out stance phase; calculate start and final timestamps.
#   #     store the duration of stance phase

#   Total_Duration = # sum duration of stance phase over the entire day
#   N = # sum step count over all sessions in the same day
#   D = # (sum of the stance phase during the day for all sesions) / N
#   H = # sum of all the session durations for the given day.


#   I = avg_F / (BW * R_max[age_category, gender])
#   E = N / H
#   P = P
#   return (I, E, D, H, P)


def calculate_RSI(params):
    """
    Calculates the Revised Strain Index (RSI) using a single list argument.

    Parameters:
    params (list or tuple): Must contain [I, E, D, H, P] in that exact order:
        I: Intensity of exertion (0.0 < I <= 1.0)
        E: Exertions per minute
        D: Duration per exertion in seconds
        H: Task duration per day in hours
        P: Posture angle in degrees (Use positive for extension, negative for flexion)
    """
    # Unpack the single list/tuple argument into the 5 variables
    I, E, D, H, P = params

    # 1. Intensity of Exertion Multiplier (IM)
    if 0.0 < I <= 0.4:
        IM = (30.00 * (I**3)) - (15.60 * (I**2)) + (13.00 * I) + 0.40
    elif 0.4 < I <= 1.0:
        IM = (36.00 * (I**3)) - (33.30 * (I**2)) + (24.77 * I) - 1.86
    else:
        raise ValueError("Intensity (I) must be between 0 and 1.")

    # 2. Exertions Per Minute Multiplier (EM)
    if E <= 90:
        EM = 0.10 + (0.25 * E)
    else:
        EM = 0.00334 * (E**1.96)

    # 3. Duration Per Exertion Multiplier (DM)
    if D <= 60:
        DM = 0.45 + (0.31 * D)
    else:
        DM = (19.17 * math.log(D)) - 59.44

    # 4. Posture Multiplier (PM)
    if P < 0:  # Negative degrees indicate Flexion
        PM = (1.2 * math.exp(0.009 * abs(P))) - 0.2
    else:     # Positive degrees indicate Extension
        if P <= 30:
            PM = 1.0
        else:
            PM = 1.0 + (0.00028 * ((P - 30)**2))

    # 5. Duration of Task Per Day Multiplier (HM)
    if H <= 0.05:
        HM = 0.20
    else:
        HM = (0.042 * H) + (0.090 * math.log(H)) + 0.477

    # Calculate final RSI score
    return IM * EM * DM * PM * HM






# Gait Classification

## Training Models

### Initialisation

In [4]:
import pandas as pd
import numpy as np
from scipy.interpolate import interp1d
from sklearn.preprocessing import StandardScaler

def prepare_training_data(patient_ids, date_list, patient_db_base_path, meta_db_path, N=100, features=['force', 'pitch', 'az']):
    """
    Loads data, normalises per stride (100 points), and performs subject-wise Z-normalisation.
    """
    processed_steps = []

    for p_id in patient_ids:
        p_db_path = f"{patient_db_base_path}/{p_id}.db"
        patient_data_list = []

        for date in date_list:
            df, meta = get_patient_daily_data(p_db_path, meta_db_path, p_id, date)
            if df is not None:
                patient_data_list.append(df)

        if not patient_data_list:
            continue

        full_p_df = pd.concat(patient_data_list)

        # Process each step
        for (sess_id, step_idx), step_df in full_p_df.groupby(['session_id', 'step_index']):
            if len(step_df) < 5: continue

            # 1. Normalise timestamps & Interpolate
            step_df = step_df.sort_values('time')
            t = step_df['time'].values
            t_norm = (t - t[0]) / (t[-1] - t[0])
            t_new = np.linspace(0, 1, N)

            step_features = {}
            for feat in features:
                f_interp = interp1d(t_norm, step_df[feat].values, kind='linear', fill_value="extrapolate")
                step_features[feat] = f_interp(t_new)

            # Add metadata/labels
            step_features['patient_id'] = p_id
            step_features['gait'] = step_df['gait'].iloc[0] if 'gait' in step_df.columns else 0
            step_features['terrain'] = step_df['terrain'].iloc[0] if 'terrain' in step_df.columns else 0
            step_features['duration'] = t[-1] - t[0]

            processed_steps.append(step_features)

    # Convert to DF for Z-normalisation
    all_steps_df = pd.DataFrame(processed_steps)

    # 4. Subject-wise Z-normalisation
    for p_id in patient_ids:
        mask = all_steps_df['patient_id'] == p_id
        if not mask.any(): continue

        for feat in features:
            # Flatten to normalise across all steps for this patient
            vals = np.stack(all_steps_df.loc[mask, feat].values)
            mean, std = np.mean(vals), np.std(vals)
            all_steps_df.loc[mask, feat] = all_steps_df.loc[mask, feat].apply(lambda x: (x - mean) / (std + 1e-6))

    return all_steps_df

### CNN

In [3]:
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA

# 1. Correlation Matrix for Redundancy Testing
def plot_correlation_matrix(train_df, features=['force', 'pitch', 'az']):
    # Flatten the 100-point arrays to check correlation across all points
    data_flat = []
    for feat in features:
        feat_data = np.stack(train_df[feat].values)
        data_flat.append(pd.DataFrame(feat_data).add_prefix(f"{feat}_"))

    combined_df = pd.concat(data_flat, axis=1)
    corr = combined_df.corr()

    plt.figure(figsize=(12, 10))
    sns.heatmap(corr, cmap='coolwarm', xticklabels=False, yticklabels=False)
    plt.title("Correlation Matrix of Normalised Stride Variables")
    plt.show()

# 2. Deep Clustering (Unsupervised Stride Grouping)
def perform_stride_clustering(train_df, n_clusters=4, features=['force', 'pitch', 'az']):
    # Prepare data: Concatenate all feature channels into one long vector per stride
    X = np.hstack([np.stack(train_df[f].values) for f in features])

    kmeans = KMeans(n_clusters=n_clusters, random_state=42, n_init=10)
    train_df['cluster'] = kmeans.fit_predict(X)

    # Dimensionality reduction for visualization
    pca = PCA(n_components=2)
    coords = pca.fit_transform(X)
    train_df['pca_1'] = coords[:, 0]
    train_df['pca_2'] = coords[:, 1]

    # Plotting
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))

    sns.scatterplot(data=train_df, x='pca_1', y='pca_2', hue='gait', palette='viridis', ax=ax1)
    ax1.set_title("Clustering Results: Color-coded by Gait Label")

    sns.scatterplot(data=train_df, x='pca_1', y='pca_2', hue='terrain', palette='magma', ax=ax2)
    ax2.set_title("Clustering Results: Color-coded by Terrain")

    plt.show()
    return train_df

# Note: To run this, you'll need the 'all_steps_df' from the previous step
# plot_correlation_matrix(all_steps_df)
# all_steps_df = perform_stride_clustering(all_steps_df)

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class GaitCNN(nn.Module):
    def __init__(self, num_features=3, sequence_length=100, num_classes=4):
        super(GaitCNN, self).__init__()
        # Input shape: (Batch, num_features, sequence_length)

        self.conv1 = nn.Conv1d(num_features, 32, kernel_size=5, padding=2)
        self.bn1 = nn.BatchNorm1d(32)
        self.conv2 = nn.Conv1d(32, 64, kernel_size=3, padding=1)
        self.bn2 = nn.BatchNorm1d(64)
        self.pool = nn.MaxPool1d(2)

        self.dropout = nn.Dropout(0.3)

        # Calculation for linear layer input:
        # 100 -> Pool(2) -> 50 -> Pool(2) -> 25
        self.fc1 = nn.Linear(64 * 25, 128)
        self.fc2 = nn.Linear(128, num_classes)

    def forward(self, x):
        x = self.pool(F.relu(self.bn1(self.conv1(x))))
        x = self.pool(F.relu(self.bn2(self.conv2(x))))

        x = torch.flatten(x, 1)
        x = F.relu(self.fc1(x))
        x = self.dropout(x)
        x = self.fc2(x)
        return x

print("CNN Architecture defined and ready for training.")

In [ ]:
class TerrainCNN(nn.Module):
    def __init__(self, num_features=3, sequence_length=100, num_classes=3):
        super(TerrainCNN, self).__init__()
        # Designed to classify terrain types (e.g., Level, Incline, Stairs)
        self.conv1 = nn.Conv1d(num_features, 32, kernel_size=5, padding=2)
        self.bn1 = nn.BatchNorm1d(32)
        self.conv2 = nn.Conv1d(32, 64, kernel_size=3, padding=1)
        self.bn2 = nn.BatchNorm1d(64)
        self.pool = nn.MaxPool1d(2)

        self.dropout = nn.Dropout(0.3)
        self.fc1 = nn.Linear(64 * 25, 128)
        self.fc2 = nn.Linear(128, num_classes)

    def forward(self, x):
        x = self.pool(F.relu(self.bn1(self.conv1(x))))
        x = self.pool(F.relu(self.bn2(self.conv2(x))))

        x = torch.flatten(x, 1)
        x = F.relu(self.fc1(x))
        x = self.dropout(x)
        x = self.fc2(x)
        return x

print("Terrain CNN Architecture defined.")

# use cross entropy loss function remember it is continuous

In [ ]:
from torch.utils.data import Dataset, DataLoader, Subset
from sklearn.model_selection import train_test_split

class GaitDataset(Dataset):
    def __init__(self, df, features=['force', 'pitch', 'az'], target_col='gait'):
        self.X = np.array([np.stack(row) for row in df[features].values]).astype(np.float32)
        # Ensure targets are 0-indexed integers
        self.y = (df[target_col].values - df[target_col].min()).astype(np.int64)

    def __len__(self):
        return len(self.y)

    def __getitem__(self, idx):
        return torch.from_numpy(self.X[idx]), torch.tensor(self.y[idx])

def train_and_evaluate_model(all_steps_df, target_col='gait', features=['force', 'pitch', 'az'], epochs=20, batch_size=32):
    # 1. Split Data (70% Train, 15% Val, 15% Test)
    train_idx, temp_idx = train_test_split(np.arange(len(all_steps_df)), test_size=0.3, random_state=42)
    val_idx, test_idx = train_test_split(temp_idx, test_size=0.5, random_state=42)

    full_dataset = GaitDataset(all_steps_df, features=features, target_col=target_col)
    train_loader = DataLoader(Subset(full_dataset, train_idx), batch_size=batch_size, shuffle=True)
    val_loader = DataLoader(Subset(full_dataset, val_idx), batch_size=batch_size)
    test_loader = DataLoader(Subset(full_dataset, test_idx), batch_size=batch_size)

    # 2. Setup Model based on target
    num_classes = all_steps_df[target_col].nunique()
    model = GaitCNN(num_features=len(features), num_classes=num_classes) if target_col == 'gait' else \
            TerrainCNN(num_features=len(features), num_classes=num_classes)

    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

    # 3. Training Loop
    for epoch in range(epochs):
        model.train()
        for inputs, labels in train_loader:
            optimizer.zero_grad()
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()

        # Validation
        model.eval()
        val_correct = 0
        with torch.no_grad():
            for inputs, labels in val_loader:
                outputs = model(inputs)
                val_correct += (outputs.argmax(1) == labels).sum().item()

        print(f"Epoch {epoch+1}/{epochs} | Val Accuracy: {val_correct/len(val_idx):.4f}")

    # 4. Final Testing
    test_correct = 0
    model.eval()
    with torch.no_grad():
        for inputs, labels in test_loader:
            outputs = model(inputs)
            test_correct += (outputs.argmax(1) == labels).sum().item()

    final_acc = test_correct / len(test_idx)
    print(f"\nFinal Test Accuracy for {target_col}: {final_acc:.4f}")
    return model

### Random Forests

In [ ]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.model_selection import train_test_split

def extract_rf_features(all_steps_df, features=['force', 'pitch', 'az']):
    rf_data = []
    for _, row in all_steps_df.iterrows():
        feat_dict = {
            'duration': row['duration'],
            'gait': row['gait'],
            'terrain': row['terrain'],
            'patient_id': row['patient_id']
        }

        for f in features:
            data = row[f]
            feat_dict[f'{f}_mean'] = np.mean(data)
            feat_dict[f'{f}_std'] = np.std(data)
            feat_dict[f'{f}_max'] = np.max(data)
            feat_dict[f'{f}_min'] = np.min(data)

        rf_data.append(feat_dict)

    return pd.DataFrame(rf_data)

# 1. Feature Extraction
rf_features_df = extract_rf_features(all_steps_df)

# 2. Correlation Matrix
plt.figure(figsize=(14, 10))
corr = rf_features_df.drop(columns=['patient_id', 'gait', 'terrain']).corr()
sns.heatmap(corr, annot=False, cmap='coolwarm')
plt.title("Correlation Matrix of Aggregated Stride Features")
plt.show()

# 3. Setup and Train Random Forest
def train_random_forest(df, target_col='gait'):
    X = df.drop(columns=['patient_id', 'gait', 'terrain'])
    y = df[target_col]

    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

    rf_model = RandomForestClassifier(n_estimators=100, random_state=42)
    rf_model.fit(X_train, y_train)

    y_pred = rf_model.predict(X_test)
    print(f"--- Results for {target_col} ---")
    print(classification_report(y_test, y_pred))

    return rf_model

# Example usage:
# gait_rf = train_random_forest(rf_features_df, target_col='gait')
# terrain_rf = train_random_forest(rf_features_df, target_col='terrain')


### Standard Inference

### Heuristic Methods

## Calling Trained Model

# Weight Bearing on Injured Leg

In [ ]:
def injured_load(df, BW, pitch_threshold=2.5):
    """
    Calculates the average weight bearing load on the injured leg for a given day.
    """
    loads = []

    for sess_id in df['session_id'].unique():
        sess_df = df[df['session_id'] == sess_id].sort_values('time')
        step_indices = sorted(sess_df['step_index'].unique())

        # Drop only first and last step (2 in total)
        if len(step_indices) <= 2: continue
        analyzed_steps = step_indices[1:-1]

        for step in analyzed_steps:
            step_df = sess_df[sess_df['step_index'] == step]

            # Filter for moments when pitch is around 0 degrees
            level_pitch_df = step_df[np.abs(step_df['pitch']) <= pitch_threshold]

            if level_pitch_df.empty: continue

            peak_force = level_pitch_df['force'].max()
            gait = step_df['gait'].iloc[0] if 'gait' in step_df.columns else None

            L = np.nan
            if gait == 1:
                L = 0
            elif gait == 2:
                L = np.abs(peak_force - BW)
            elif gait in [3, 4]:
                L = np.abs(2 * peak_force - BW)

            if not np.isnan(L):
                loads.append(L)

    return np.mean(loads) if loads else 0.0

# Fall Detection